TASK 1: INGEST (ingest.py)

Nhiệm vụ: Đọc file từ Volume landing bằng Auto Loader vào Bronze, đếm số dòng và truyền số liệu qua taskValues.

In [0]:
from pyspark.sql.functions import col, current_timestamp

TEN = "duykh"
BASE_DIR   = f"/Volumes/workspace/lab_{TEN}"
LANDING    = f"{BASE_DIR}/raw/landing"
CKPT_DIR   = f"{BASE_DIR}/_ckpt/orders_ckpt"
SCHEMA_DIR = f"{BASE_DIR}/_ckpt/orders_schema"
TARGET_TBL = f"workspace.bronze_{TEN}.raw_orders_{TEN}"

# 1. Auto Loader đọc stream tăng dần từ Volume nội bộ
df_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("cloudFiles.schemaLocation", SCHEMA_DIR)
         .option("cloudFiles.inferColumnTypes", "true")
         .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
         .option("header", "true")
         .load(LANDING)
)

df_enriched = (
    df_stream
      .withColumn("_ingested_at", current_timestamp())
      .withColumn("_source_file", col("_metadata.file_path"))
)

query = (
    df_enriched.writeStream
      .format("delta")
      .option("checkpointLocation", CKPT_DIR)
      .option("mergeSchema", "true")
      .trigger(availableNow=True)
      .toTable(TARGET_TBL)
)
query.awaitTermination()

# 3. Đếm số dòng Bronze và ghi vào taskValues
bronze_count = spark.table(TARGET_TBL).count()
print(f"Tổng số dòng hiện tại trong Bronze ({TARGET_TBL}): {bronze_count}")

# BẮT BUỘC: Đẩy số liệu sang cho task sau
dbutils.jobs.taskValues.set(key="bronze_rows", value=bronze_count)
print("Đã ghi taskValues: bronze_rows =", bronze_count)
